In [0]:
from LDCDataAccessLayerPy import KeyVaultManager, SharePointManager, SqlManager, databricks_init
from datetime import datetime, timedelta
from LDCDataAccessLayerPy import databricks_init, DataLakeManagerGen2
from io import BytesIO
import LDCDataAccessLayerPy
#Initiate the secret to access KeyVault secrets
databricks_init(dbutils, 'GO')
sp_mgr = SharePointManager()
sql_mgr = SqlManager()

import logging
logger = spark._jvm.org.apache.log4j
logging.getLogger("py4j").setLevel(logging.ERROR)

import pandas as pd
import os
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib.colors import ListedColormap
from sklearn.cluster import KMeans
from openpyxl import load_workbook

from datetime import datetime, timedelta
import re
from dateutil.relativedelta import relativedelta
from LDCDataAccessLayerPy import PriceManager, graph
from LDCDataAccessLayerPy import ZemaManager
import math
import plotly.express as px

zema = ZemaManager()


In [0]:
start_date = datetime(2020, 1, 1)

end_date = datetime(2030, 1, 1)
url = "https://ldcom365.sharepoint.com"

In [0]:
def assign_season_corn(row):
    date = row['date']
    if date.month >= 3:  # March to December → same year
        season_start = date.year
    else:  # January, February → previous year's marketing season
        season_start = date.year - 1
    return f"{season_start}/{season_start + 1}"
  
def adjust_virtual_date_corn(row):
    if row['virtual_date'].month in [1, 2]:
        # Subtract 1 year from the year if month is November or December
        return row['virtual_date']
    else:
        # Keep the date as is if the month is not November or December
        return row['virtual_date'].replace(year=row['virtual_date'].year - 1)

### CBOT

### UPR

In [0]:
prem_corn= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
prem_corn=prem_corn[prem_corn['observation']=='Last']
prem_corn=prem_corn[['date','value','contract_year','contract_month']]
prem_corn['day']=prem_corn['date'].dt.day
prem_corn['month']=prem_corn['date'].dt.month
prem_corn['year']=prem_corn['date'].dt.year
prem_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': prem_corn['month'], 'day': prem_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
prem_corn = prem_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
prem_corn=prem_corn[['date','value','contract_year','contract_month','virtual_date']]

prem_corn['contract_date'] = pd.to_datetime(dict(year=prem_corn['contract_year'], month=prem_corn['contract_month'], day=1))

prem_corn['virtual_date'] = prem_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
prem_corn['day'] = prem_corn['date'].dt.day
prem_corn['month'] = prem_corn['date'].dt.month
prem_corn['year'] = prem_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
prem_corn[['target_month', 'target_year']] = prem_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_prem_corn = prem_corn[(prem_corn['contract_month'] == prem_corn['target_month']) & 
             (prem_corn['contract_year'] == prem_corn['target_year'])]

spot_prem_corn=spot_prem_corn.rename(columns={'value': 'UPR'})
spot_prem_corn=spot_prem_corn[['date','UPR','virtual_date']]


### BB

In [0]:
prem_corn_bb= zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Bahia Blanca Arg-USDc-Bu", period=f"{start_date}::{end_date}")
prem_corn_bb=prem_corn_bb[prem_corn_bb['observation']=='Last']
prem_corn_bb=prem_corn_bb[['date','value','contract_year','contract_month']]
prem_corn_bb['day']=prem_corn_bb['date'].dt.day
prem_corn_bb['month']=prem_corn_bb['date'].dt.month
prem_corn_bb['year']=prem_corn_bb['date'].dt.year
prem_corn_bb['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': prem_corn_bb['month'], 'day': prem_corn_bb['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
prem_corn_bb = prem_corn_bb.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
prem_corn_bb=prem_corn_bb[['date','value','contract_year','contract_month','virtual_date']]

prem_corn_bb['contract_date'] = pd.to_datetime(dict(year=prem_corn_bb['contract_year'], month=prem_corn_bb['contract_month'], day=1))

prem_corn_bb['virtual_date'] = prem_corn_bb.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
prem_corn_bb['day'] = prem_corn_bb['date'].dt.day
prem_corn_bb['month'] = prem_corn_bb['date'].dt.month
prem_corn_bb['year'] = prem_corn_bb['date'].dt.year

# Apply to get target contract month and year
prem_corn_bb[['target_month', 'target_year']] = prem_corn_bb.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_prem_corn_bb = prem_corn_bb[(prem_corn_bb['contract_month'] == prem_corn_bb['target_month']) & 
             (prem_corn_bb['contract_year'] == prem_corn_bb['target_year'])]

spot_prem_corn_bb=spot_prem_corn_bb.rename(columns={'value': 'bb'})
spot_prem_corn_bb=spot_prem_corn_bb[['date','bb']]

### UPBB

In [0]:
corn_upbb=pd.merge(spot_prem_corn,spot_prem_corn_bb,on='date')
corn_upbb['UPBB']=corn_upbb['UPR']*0.65+corn_upbb['bb']*0.35
corn_upbb['date']=pd.to_datetime(corn_upbb['date'])
corn_upbb['Season']= corn_upbb.apply(assign_season_corn, axis=1)

In [0]:
# Plot with plotly express
fig = px.line(
    corn_upbb,
    x='virtual_date',
    y='UPBB',
    color='Season',
    title='Value over Virtual Date by Season',
    labels={'value': 'Value', 'virtual_date': 'Virtual Date'}
)

fig.update_layout(
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

### USG

In [0]:
usg_corn=zema.get_curve(curve="P-CASH-LDC-INPUT-PREMIUM-CORN-AR-FOB Up River Arg-USDc-Bu", period=f"{start_date}::{end_date}")
usg_corn=usg_corn[usg_corn['observation']=='Last']
usg_corn=usg_corn[['date','value','contract_year','contract_month']]
usg_corn['day']=usg_corn['date'].dt.day
usg_corn['month']=usg_corn['date'].dt.month
usg_corn['year']=usg_corn['date'].dt.year
usg_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': usg_corn['month'], 'day': usg_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
usg_corn = usg_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
usg_corn=usg_corn[['date','value','contract_year','contract_month','virtual_date']]

usg_corn['contract_date'] = pd.to_datetime(dict(year=usg_corn['contract_year'], month=usg_corn['contract_month'], day=1))

usg_corn['virtual_date'] = usg_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
usg_corn['day'] = usg_corn['date'].dt.day
usg_corn['month'] = usg_corn['date'].dt.month
usg_corn['year'] = usg_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
usg_corn[['target_month', 'target_year']] = usg_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_usg_corn = usg_corn[(usg_corn['contract_month'] == usg_corn['target_month']) & 
             (usg_corn['contract_year'] == usg_corn['target_year'])]

# Drop helper columns if not needed
spot_usg_corn = spot_usg_corn.drop(columns=['day', 'month', 'year', 'target_month', 'target_year'])

### PNW

In [0]:
pnw_corn=zema.get_curve(curve="P-CASH-LDC-CALC-PREMIUM-CORN-US-FOB-FBV-PNW-YELLOW-USDc-Bu", period=f"{start_date}::{end_date}")
pnw_corn=pnw_corn[pnw_corn['observation']=='Last']
pnw_corn=pnw_corn[['date','value','contract_year','contract_month']]
pnw_corn['day']=pnw_corn['date'].dt.day
pnw_corn['month']=pnw_corn['date'].dt.month
pnw_corn['year']=pnw_corn['date'].dt.year
pnw_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': pnw_corn['month'], 'day': pnw_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
pnw_corn = pnw_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
pnw_corn=pnw_corn[['date','value','contract_year','contract_month','virtual_date']]

pnw_corn['contract_date'] = pd.to_datetime(dict(year=pnw_corn['contract_year'], month=pnw_corn['contract_month'], day=1))

pnw_corn['virtual_date'] = pnw_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
pnw_corn['day'] = pnw_corn['date'].dt.day
pnw_corn['month'] = pnw_corn['date'].dt.month
pnw_corn['year'] = pnw_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
pnw_corn[['target_month', 'target_year']] = pnw_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_pnw_corn = pnw_corn[(pnw_corn['contract_month'] == pnw_corn['target_month']) & 
             (pnw_corn['contract_year'] == pnw_corn['target_year'])]

# Drop helper columns if not needed
spot_pnw_corn = spot_pnw_corn.drop(columns=['day', 'month', 'year', 'target_month', 'target_year'])

In [0]:
spot_pnw_corn_df=spot_pnw_corn.rename(columns={'value':'FOB_PNW'})
spot_usg_corn_df=spot_usg_corn.rename(columns={'value':'FOB_USG'})
corn_upbb_df=corn_upbb.rename(columns={'UPBB':'FOB_UPBB'})

spot_pnw_corn_df=spot_pnw_corn_df[['date','FOB_PNW','virtual_date']]
spot_usg_corn_df=spot_usg_corn_df[['date','FOB_USG']]
corn_upbb_df=corn_upbb_df[['date','FOB_UPBB','Season']]

fob_merged=pd.merge(spot_pnw_corn_df,spot_usg_corn_df,on='date')
fob_merged=pd.merge(fob_merged,corn_upbb_df,on='date')
fob_merged


In [0]:
import plotly.graph_objects as go
import plotly.express as px

# Define line styles for each FOB
line_styles = {
    'FOB_PNW': 'solid',
    'FOB_USG': 'dot',
    'FOB_UPBB': 'dash'
}

# Assign unique color to each season (excluding 2022/2023)
season_list = fob_merged['Season'].unique()
season_colors = {
    season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
    for i, season in enumerate(season_list) if season != '2022/2023'
}

# Build figure
fob = go.Figure()

for season in season_list:
    if season == '2022/2023':
        continue
    df_season = fob_merged[fob_merged['Season'] == season]

    for col, style in line_styles.items():
        fob.add_trace(go.Scatter(
            x=df_season['virtual_date'],
            y=df_season[col],
            mode='lines',
            name=f"{col} ({season})",
            line=dict(dash=style, color=season_colors[season])
        ))

# Layout
fob.update_layout(
    title='FOB PNW, USG, UPBB by Season',
    xaxis=dict(title='Virtual Date', tickformat="%b %d"),
    yaxis=dict(title='FOB (USD/mt)'),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

fob.show()

# Optional HTML export
fob_html = fob.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
fob_prices  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to Korea</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>FOB PRICES</h1>

    <div class="chart-container">{fob}</div>
</body>
</html>
"""
FOB_report_bytes = fob_html.encode("utf-8")

test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# # Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=test_2,
    subject=f'FOB values {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body="Please find the attached report.", attachment={"FOB_Values.html": FOB_report_bytes}
)



## FREIGHT

In [0]:
arg_sa=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-CFR-UP(10.36m fw)+BB -SA-Dammam-PANAMAX-V000003188-USD-MT", period=f"{start_date}::{end_date}")
arg_viet=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-CFR-UP(10.36m fw)+BB -VN-Cai Mep-PANAMAX-V000003325-USD-MT", period=f"{start_date}::{end_date}")
arg_kor=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-CFR-UP(10.36m fw)+BB -KR-Incheon-PANAMAX-V000003327-USD-MT", period=f"{start_date}::{end_date}")
arg_egy=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-CFR-UP(10.36m fw)+BB -EG-El Dekheila-PANAMAX-V000002631-USD-MT", period=f"{start_date}::{end_date}")

us_sa=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-CFR-PORT ALLEN-SA-YANBU-VIA SUEZ-YELLOW-PANAMAX-V000008846-USD-MT", period=f"{start_date}::{end_date}")
us_viet=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-CFR-Seattle-VN-Cai Mep-PANAMAX-USD-MT", period=f"{start_date}::{end_date}")
us_kor=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-CFR-Seattle-KR-Incheon-PANAMAX-USD-MT", period=f"{start_date}::{end_date}")
us_egy=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-CFR-Port Allen-EG-El Dekheila-via Suez-PANAMAX-USD-MT", period=f"{start_date}::{end_date}")


In [0]:
dfs=[arg_sa,arg_viet,arg_kor,arg_egy,us_sa,us_viet,us_kor,us_egy]

In [0]:
# Define corresponding names for clarity in outputs
df_names = ['arg_sa', 'arg_viet', 'arg_kor','arg_egy', 'us_sa','us_viet', 'us_kor','us_egy']

# Function to process each DataFrame
def process_df(df):
    df = df[df['observation'] == 'Last']
    df = df[['date', 'value', 'contract_year', 'contract_month']]

    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    df['virtual_date'] = pd.to_datetime(
        {'year': 2000, 'month': df['month'], 'day': df['day']},
        errors='coerce'
    )

    df = df.dropna(subset=['virtual_date'])

    df = df[['date', 'value', 'contract_year', 'contract_month', 'virtual_date']]

    df['contract_date'] = pd.to_datetime(
        dict(year=df['contract_year'], month=df['contract_month'], day=1)
    )

    df['virtual_date'] = df.apply(adjust_virtual_date_corn, axis=1)

    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year

    df[['target_month', 'target_year']] = df.apply(get_spot_contract_info, axis=1)

    spot_df = df[
        (df['contract_month'] == df['target_month']) &
        (df['contract_year'] == df['target_year'])
    ]

    spot_df = spot_df.drop(columns=['day', 'month', 'year', 'target_month', 'target_year'])

    return spot_df

# Dictionary to store results
spot_dfs = {}

# Loop through your list
for df, name in zip(dfs, df_names):
    spot_dfs[f'spot_{name}'] = process_df(df)


In [0]:
spot_arg_sa = spot_dfs['spot_arg_sa']
spot_arg_viet = spot_dfs['spot_arg_viet']
spot_arg_kor = spot_dfs['spot_arg_kor']
spot_arg_egy = spot_dfs['spot_arg_egy']
spot_us_viet = spot_dfs['spot_us_viet']
spot_us_kor = spot_dfs['spot_us_kor']
spot_us_sa = spot_dfs['spot_us_sa']
spot_us_egy = spot_dfs['spot_us_egy']


### WITH KOREA

In [0]:
import plotly.graph_objects as go

In [0]:
spot_arg_kor = spot_arg_kor.rename(columns={'value': 'arg_to_kor','virtual_date':'virtual'})
spot_us_kor = spot_us_kor.rename(columns={'value': 'us_to_kor'})
freight_kor=pd.merge(spot_us_kor,spot_arg_kor,on='date',how='inner')
freight_kor=freight_kor[['date','arg_to_kor','us_to_kor','virtual']]
freight_kor['spread']=(freight_kor['arg_to_kor']-freight_kor['us_to_kor'])/0.3937

prem_w_freight=pd.merge(corn_upbb,freight_kor,on='date',how='inner')
prem_w_freight['arg_eq_us']=prem_w_freight['UPBB']+prem_w_freight['spread']


prem_w_freight=prem_w_freight[['date','UPBB','arg_eq_us','virtual','spread','Season']]
prem_w_freight_w_us=pd.merge(prem_w_freight,spot_pnw_corn,on='date',how='inner')

prem_w_freight_w_us['final_spread']=prem_w_freight_w_us['arg_eq_us']-prem_w_freight_w_us['value']
# Plot with plotly express
kor = px.line(
    prem_w_freight_w_us,
    x='virtual_date',
    y='final_spread',
    color='Season',
    title='Arg Corn to Korea',
    labels={'final_spread': 'Value'}
)


kor.update_layout(
    template='plotly_white',
    hovermode='x unified'
)

kor.show()

In [0]:
import plotly.graph_objects as go

kor = go.Figure()

seasons = prem_w_freight_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season

    df_season = prem_w_freight_w_us[prem_w_freight_w_us['Season'] == season]

    # Final spread (left axis)
    kor.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    kor.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'US PNW ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    kor.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
kor.update_layout(
    title='Arg Corn to Korea — Spread vs US PNW by Season',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (USD)', side='left'),
    yaxis2=dict(
        title='US PNW Value (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

kor.show()
kor_html = kor.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
Korea_arg_us  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to Korea</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to Korea, ARG VS US</h1>

    <div class="chart-container">{kor_html}</div>
</body>
</html>
"""
Korea_report_bytes = kor_html.encode("utf-8")

test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test_2,
#     subject=f'CORN TO KOREA   {datetime.now().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body="Please find the attached report.", attachment={"KOREA_Values.html": Korea_report_bytes}
# )



## WITH VIETNAM

In [0]:
spot_arg_viet = spot_arg_viet.rename(columns={'value': 'arg_to_viet','virtual_date':'virtual'})
spot_us_viet = spot_us_viet.rename(columns={'value': 'us_to_viet'})
freight_viet=pd.merge(spot_us_viet,spot_arg_viet,on='date',how='inner')
freight_viet=freight_viet[['date','arg_to_viet','us_to_viet','virtual']]
freight_viet['spread']=(freight_viet['arg_to_viet']-freight_viet['us_to_viet'])/0.3937

viet_w_freight=pd.merge(corn_upbb,freight_viet,on='date',how='inner')
viet_w_freight['arg_eq_us']=viet_w_freight['UPBB']+viet_w_freight['spread']


viet_w_freight=viet_w_freight[['date','UPBB','arg_eq_us','virtual','spread','Season']]
viet_w_freight_w_us=pd.merge(viet_w_freight,spot_pnw_corn,on='date',how='inner')

viet_w_freight_w_us['final_spread']=viet_w_freight_w_us['arg_eq_us']-viet_w_freight_w_us['value']
# Plot with plotly express
viet = px.line(
    viet_w_freight_w_us,
    x='virtual_date',
    y='final_spread',
    color='Season',
    title='Arg Corn to Vietnam vs US',
    labels={'final_spread': 'Value'}
)

viet.update_layout(
    template='plotly_white',
    hovermode='x unified'
)

viet.show()

In [0]:
import plotly.graph_objects as go


viet = go.Figure()

seasons = viet_w_freight_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season
    df_season = viet_w_freight_w_us[viet_w_freight_w_us['Season'] == season]

    # Final spread (left axis)
    viet.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    viet.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'US PNW ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    viet.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
viet.update_layout(
    title='Arg Corn to Vietnam — Spread vs US PNW by Season',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (USD)', side='left'),
    yaxis2=dict(
        title='US PNW Value (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

viet.show()
viet_html = viet.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
Viet_arg_us  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to Viet</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to Viet, ARG VS US</h1>

    <div class="chart-container">{viet}</div>
</body>
</html>
"""
Viet_report_bytes = viet_html.encode("utf-8")

In [0]:

test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test,
#     subject=f'CORN TO Vietnam   {datetime.now().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body="Please find the attached report.", attachment={"Vietnam_Values.html": Viet_report_bytes}
# )



### SAUDI

In [0]:
spot_arg_sa = spot_arg_sa.rename(columns={'value': 'arg_to_sa','virtual_date':'virtual'})
spot_us_sa = spot_us_sa.rename(columns={'value': 'us_to_sa'})
freight_sa=pd.merge(spot_us_sa,spot_arg_sa,on='date',how='inner')
freight_sa=freight_sa[['date','arg_to_sa','us_to_sa','virtual']]
freight_sa['spread']=(freight_sa['arg_to_sa']-freight_sa['us_to_sa'])/0.3937

sa_w_freight=pd.merge(corn_upbb,freight_sa,on='date',how='inner')
sa_w_freight['arg_eq_us']=sa_w_freight['UPBB']+sa_w_freight['spread']


sa_w_freight=sa_w_freight[['date','UPBB','arg_eq_us','virtual','spread','Season']]
sa_w_freight_w_us=pd.merge(sa_w_freight,spot_pnw_corn,on='date',how='inner')

sa_w_freight_w_us['final_spread']=sa_w_freight_w_us['arg_eq_us']-sa_w_freight_w_us['value']
# Plot with plotly express
sa = px.line(
    sa_w_freight_w_us,
    x='virtual_date',
    y='final_spread',
    color='Season',
    title='Arg Corn to Damman vs US to Yambu',
    labels={'final_spread': 'Value'}
)

sa.update_layout(
    template='plotly_white',
    hovermode='x unified'
)

sa.show()

In [0]:
import plotly.graph_objects as go

sa = go.Figure()

seasons = sa_w_freight_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season
    df_season = sa_w_freight_w_us[sa_w_freight_w_us['Season'] == season]

    # Final spread (left axis)
    sa.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    sa.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'USG ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    sa.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
sa.update_layout(
    title='Arg Corn to Dammam VS USG to Yanbu',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (cts/bu)', side='left'),
    yaxis2=dict(
        title='USG Value',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

sa.show()
sa_html = sa.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
Sa_arg_us  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to Sa</title>
    <style>
        body {{
            font-family: Arial, sans-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to Saudi, ARG VS US</h1>

    <div class="chart-container">{sa}</div>
</body>
</html>
"""
Sa_report_bytes = sa_html.encode("utf-8")

In [0]:

test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test_2,
#     subject=f'CORN TO Saudi arabia {datetime.now().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body="Please find the attached report.", attachment={"Saudi_Values.html": Sa_report_bytes}
# )



### EGYPT

In [0]:
spot_arg_egy = spot_arg_egy.rename(columns={'value': 'arg_to_egy','virtual_date':'virtual'})
spot_us_egy = spot_us_egy.rename(columns={'value': 'us_to_egy'})
freight_egy=pd.merge(spot_us_egy,spot_arg_egy,on='date',how='inner')
freight_egy=freight_egy[['date','arg_to_egy','us_to_egy','virtual']]
freight_egy['spread']=(freight_egy['arg_to_egy']-freight_egy['us_to_egy'])/0.3937

egy_w_freight=pd.merge(corn_upbb,freight_egy,on='date',how='inner')
egy_w_freight['arg_eq_us']=egy_w_freight['UPBB']+egy_w_freight['spread']


egy_w_freight=egy_w_freight[['date','UPBB','arg_eq_us','virtual','spread','Season']]
egy_w_freight_w_us=pd.merge(egy_w_freight,spot_usg_corn,on='date',how='inner')

egy_w_freight_w_us['final_spread']=egy_w_freight_w_us['arg_eq_us']-egy_w_freight_w_us['value']
# Plot with plotly express
egy = px.line(
    egy_w_freight_w_us,
    x='virtual_date',
    y='final_spread',
    color='Season',
    title='Arg Corn to Damman vs US to Yambu',
    labels={'final_spread': 'Value'}
)

egy.update_layout(
    template='plotly_white',
    hovermode='x unified'
)

egy.show()

In [0]:
import plotly.graph_objects as go

egy = go.Figure()

seasons = egy_w_freight_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season
    df_season = egy_w_freight_w_us[egy_w_freight_w_us['Season'] == season]

    # Final spread (left axis)
    egy.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    egy.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'USG ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    egy.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
egy.update_layout(
    title='Arg Corn VS USG to Egypt (El Dekheila)',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (cts/bu)', side='left'),
    yaxis2=dict(
        title='USG Value',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

egy.show()
egy_html = egy.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
egy_arg_us  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to egy</title>
    <style>
        body {{
            font-family: Arial, egyns-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to egyudi, ARG VS US</h1>

    <div class="chart-container">{egy}</div>
</body>
</html>
"""
egy_report_bytes = egy_html.encode("utf-8")

In [0]:

test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# # Send email with embedded chart and table
# LDCDataAccessLayerPy.mail.mail_send(
#     to=test_2,
#     subject=f'CORN TO Egypt {datetime.now().strftime("%d-%m")}',
#     from_addr="florian.girardi-ext@ldc.com",
#     mime_type="html",
#     body="Please find the attached report.", attachment={"Egypt_Values.html": egy_report_bytes}
# )



In [0]:
egy_w_freight_w_us=egy_w_freight_w_us.rename(columns={'final_spread': 'corn_to_egy'})
viet_w_freight_w_us=viet_w_freight_w_us.rename(columns={'final_spread': 'corn_to_viet'})
prem_w_freight_w_us=prem_w_freight_w_us.rename(columns={'final_spread': 'corn_to_kor'})

viet_w_freight_w_us_small=viet_w_freight_w_us[['date','corn_to_viet']]
egy_w_freight_w_us_small=egy_w_freight_w_us[['date','corn_to_egy']]
prem_w_freight_w_us_small=prem_w_freight_w_us[['date','corn_to_kor','virtual_date','Season']]

dest_merged=pd.merge(viet_w_freight_w_us_small,egy_w_freight_w_us_small,on='date',how='inner')
dest_merged=pd.merge(dest_merged,prem_w_freight_w_us_small,on='date',how='inner')

fob_merged_filt=fob_merged[['date','FOB_UPBB','FOB_USG','FOB_PNW']]
dest_w_fob=pd.merge(dest_merged,fob_merged_filt,on='date',how='inner')


# Define line styles for each variable
line_styles = {
    'corn_to_viet': 'solid',
    'corn_to_egy': 'dot',
    'corn_to_kor': 'dash',
    'FOB_UPBB': 'solid',
    'FOB_USG': 'dot',
    'FOB_PNW': 'dash'
}

# Separate by y-axis
spread_vars = ['corn_to_viet', 'corn_to_egy', 'corn_to_kor']
fob_vars = ['FOB_UPBB', 'FOB_USG', 'FOB_PNW']

# Assign a color to each season (excluding 2022/2023)
season_list = dest_w_fob['Season'].unique()
season_colors = {
    season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
    for i, season in enumerate(season_list) if season != '2022/2023'
}

# Initialize figure
destinations = go.Figure()

fob_colors = {
    'FOB_UPBB': 'black',
    'FOB_USG': 'red',
    'FOB_PNW': 'yellow'
}

# Add traces
for season in season_list:
    if season == '2022/2023':
        continue
    dest_w_fob_season = dest_w_fob[dest_w_fob['Season'] == season]
    color = season_colors[season]

    # Add spread variables to left y-axis using season colors
    for var in spread_vars:
        destinations.add_trace(go.Scatter(
            x=dest_w_fob_season['virtual_date'],
            y=dest_w_fob_season[var],
            name=f"{var.replace('corn_to_', '').capitalize()} ({season})",
            line=dict(color=color, dash=line_styles[var]),
            yaxis='y1'
        ))

    # Add FOB variables to right y-axis using fixed colors per FOB variable
    for var in fob_vars:
        destinations.add_trace(go.Scatter(
            x=dest_w_fob_season['virtual_date'],
            y=dest_w_fob_season[var],
            name=f"{var} ({season})",
            line=dict(color=fob_colors[var], dash=line_styles[var]),
            yaxis='y2'
        ))

# Layout
destinations.update_layout(
    title='Corn Spreads and FOB Values by Season',
    xaxis=dict(title='Virtual Date', tickformat="%b %d"),
    yaxis=dict(title='Spread (cts/bu)', side='left'),
    yaxis2=dict(
        title='FOB Price (USD/mt)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)', borderwidth=0),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

destinations.show()
destinations_html = destinations.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
# import plotly.graph_objects as go

# destinations = go.Figure()

# # Define line styles for each destination
# line_styles = {
#     'corn_to_viet': 'solid',
#     'corn_to_egy': 'dot',
#     'corn_to_kor': 'dash'
# }

# # Optional: assign a unique color per season
# import plotly.express as px
# season_list = dest_merged['Season'].unique()
# season_colors = {season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
#                  for i, season in enumerate(season_list) if season != '2022/2023'}

# # Loop through each season and add lines for each destination
# for season in season_list:
#     if season == '2022/2023':
#         continue  # Skip this season
#     dest_merged_season = dest_merged[dest_merged['Season'] == season]

#     for dest, style in line_styles.items():
#         destinations.add_trace(go.Scatter(
#             x=dest_merged_season['virtual_date'],
#             y=dest_merged_season[dest],
#             mode='lines',
#             name=f"{dest.replace('corn_to_', '').capitalize()} ({season})",
#             line=dict(dash=style, color=season_colors[season]),
#         ))

# # Update layout
# destinations.update_layout(
#     title='Arg Corn to Destinations (Spread by Season)',
#     xaxis=dict(title='Virtual Date', tickformat="%b %d"),
#     yaxis=dict(title='Spread (cts/bu)'),
#     template='plotly_white',
#     legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
#     hovermode='x unified',
#     margin=dict(l=60, r=60, t=50, b=40)
# )

# destinations.show()

# # Optional: export to HTML
# destinations_html = destinations.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:

dest_arg_us  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to egy</title>
    <style>
        body {{
            font-family: Arial, egyns-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to egyudi, ARG VS US</h1>

    <div class="chart-container">{destinations}</div>
</body>
</html>
"""
dest_report_bytes = destinations_html.encode("utf-8")


test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=test_2,
    subject=f'Corn to 3 Destinations with FOB  {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body="Please find the attached report.", attachment={"3_dest_Values.html": dest_report_bytes}
)



# WITH FLATE PRICES

### BB FLat

In [0]:
flat_bb_corn= zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Bahia Blanca Arg-USD-MT", period=f"{start_date}::{end_date}")
flat_bb_corn=flat_bb_corn[flat_bb_corn['observation']=='Last']
flat_bb_corn=flat_bb_corn[['date','value','contract_year','contract_month']]
flat_bb_corn['day']=flat_bb_corn['date'].dt.day
flat_bb_corn['month']=flat_bb_corn['date'].dt.month
flat_bb_corn['year']=flat_bb_corn['date'].dt.year
flat_bb_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': flat_bb_corn['month'], 'day': flat_bb_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
flat_bb_corn = flat_bb_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
flat_bb_corn=flat_bb_corn[['date','value','contract_year','contract_month','virtual_date']]

flat_bb_corn['contract_date'] = pd.to_datetime(dict(year=flat_bb_corn['contract_year'], month=flat_bb_corn['contract_month'], day=1))

flat_bb_corn['virtual_date'] = flat_bb_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
flat_bb_corn['day'] = flat_bb_corn['date'].dt.day
flat_bb_corn['month'] = flat_bb_corn['date'].dt.month
flat_bb_corn['year'] = flat_bb_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
flat_bb_corn[['target_month', 'target_year']] = flat_bb_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_flat_bb_corn = flat_bb_corn[(flat_bb_corn['contract_month'] == flat_bb_corn['target_month']) & 
             (flat_bb_corn['contract_year'] == flat_bb_corn['target_year'])]

spot_flat_bb_corn=spot_flat_bb_corn.rename(columns={'value': 'BB'})
spot_flat_bb_corn=spot_flat_bb_corn[['date','BB','virtual_date']]


In [0]:
flat_upr_corn= zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-AR-FOB Up River Arg-USD-MT", period=f"{start_date}::{end_date}")
flat_upr_corn=flat_upr_corn[flat_upr_corn['observation']=='Last']
flat_upr_corn=flat_upr_corn[['date','value','contract_year','contract_month']]
flat_upr_corn['day']=flat_upr_corn['date'].dt.day
flat_upr_corn['month']=flat_upr_corn['date'].dt.month
flat_upr_corn['year']=flat_upr_corn['date'].dt.year
flat_upr_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': flat_upr_corn['month'], 'day': flat_upr_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
flat_upr_corn = flat_upr_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
flat_upr_corn=flat_upr_corn[['date','value','contract_year','contract_month','virtual_date']]

flat_upr_corn['contract_date'] = pd.to_datetime(dict(year=flat_upr_corn['contract_year'], month=flat_upr_corn['contract_month'], day=1))

flat_upr_corn['virtual_date'] = flat_upr_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
flat_upr_corn['day'] = flat_upr_corn['date'].dt.day
flat_upr_corn['month'] = flat_upr_corn['date'].dt.month
flat_upr_corn['year'] = flat_upr_corn['date'].dt.year

# Apply to get target contract month and year
flat_upr_corn[['target_month', 'target_year']] = flat_upr_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_flat_upr_corn = flat_upr_corn[(flat_upr_corn['contract_month'] == flat_upr_corn['target_month']) & 
             (flat_upr_corn['contract_year'] == flat_upr_corn['target_year'])]

spot_flat_upr_corn=spot_flat_upr_corn.rename(columns={'value': 'UPR'})
spot_flat_upr_corn=spot_flat_upr_corn[['date','UPR']]

In [0]:
flat_corn_upbb=pd.merge(spot_flat_upr_corn,spot_flat_bb_corn,on='date')
flat_corn_upbb['UPBB']=flat_corn_upbb['UPR']*0.65+flat_corn_upbb['BB']*0.35
flat_corn_upbb['date']=pd.to_datetime(flat_corn_upbb['date'])
flat_corn_upbb['Season']= flat_corn_upbb.apply(assign_season_corn, axis=1)
flat_corn_upbb['UPBB']=flat_corn_upbb['UPBB']/ 0.39368

### USG

In [0]:
usg_flat_corn=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-FOB-FBV-CGF-YELLOW-USDc-Bu", period=f"{start_date}::{end_date}")
usg_flat_corn=usg_flat_corn[usg_flat_corn['observation']=='Last']
usg_flat_corn=usg_flat_corn[['date','value','contract_year','contract_month']]
usg_flat_corn['day']=usg_flat_corn['date'].dt.day
usg_flat_corn['month']=usg_flat_corn['date'].dt.month
usg_flat_corn['year']=usg_flat_corn['date'].dt.year
usg_flat_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': usg_flat_corn['month'], 'day': usg_flat_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
usg_flat_corn = usg_flat_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
usg_flat_corn=usg_flat_corn[['date','value','contract_year','contract_month','virtual_date']]

usg_flat_corn['contract_date'] = pd.to_datetime(dict(year=usg_flat_corn['contract_year'], month=usg_flat_corn['contract_month'], day=1))

usg_flat_corn['virtual_date'] = usg_flat_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
usg_flat_corn['day'] = usg_flat_corn['date'].dt.day
usg_flat_corn['month'] = usg_flat_corn['date'].dt.month
usg_flat_corn['year'] = usg_flat_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
usg_flat_corn[['target_month', 'target_year']] = usg_flat_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_usg_flat_corn = usg_flat_corn[(usg_flat_corn['contract_month'] == usg_flat_corn['target_month']) & 
             (usg_flat_corn['contract_year'] == usg_flat_corn['target_year'])]

# Drop helper columns if not needed
spot_usg_flat_corn = spot_usg_flat_corn.drop(columns=['day', 'month', 'year', 'target_month', 'target_year'])

### PNW

In [0]:
pnw_flat_corn=zema.get_curve(curve="P-CASH-LDC-CALC-FLAT-CORN-US-FOB-FBV-PNW-YELLOW-USDc-Bu", period=f"{start_date}::{end_date}")
pnw_flat_corn=pnw_flat_corn[pnw_flat_corn['observation']=='Last']
pnw_flat_corn=pnw_flat_corn[['date','value','contract_year','contract_month']]
pnw_flat_corn['day']=pnw_flat_corn['date'].dt.day
pnw_flat_corn['month']=pnw_flat_corn['date'].dt.month
pnw_flat_corn['year']=pnw_flat_corn['date'].dt.year
pnw_flat_corn['virtual_date'] = pd.to_datetime(
    {'year': 2000, 'month': pnw_flat_corn['month'], 'day': pnw_flat_corn['day']},
    errors='coerce'  # This will convert invalid dates (e.g. Feb 30) to NaT
)

# Drop rows with invalid virtual dates
pnw_flat_corn = pnw_flat_corn.dropna(subset=['virtual_date'])

# You can now sort or use this date for seasonal charts
pnw_flat_corn=pnw_flat_corn[['date','value','contract_year','contract_month','virtual_date']]

pnw_flat_corn['contract_date'] = pd.to_datetime(dict(year=pnw_flat_corn['contract_year'], month=pnw_flat_corn['contract_month'], day=1))

pnw_flat_corn['virtual_date'] = pnw_flat_corn.apply(adjust_virtual_date_corn, axis=1)

# Extract day and month
pnw_flat_corn['day'] = pnw_flat_corn['date'].dt.day
pnw_flat_corn['month'] = pnw_flat_corn['date'].dt.month
pnw_flat_corn['year'] = pnw_flat_corn['date'].dt.year

# Define target contract month/year based on the date rules
def get_spot_contract_info(row):
    if row['day'] < 15:
        target_month = row['month']
        target_year = row['year']
    else:
        if row['month'] == 12:
            target_month = 1
            target_year = row['year'] + 1
        else:
            target_month = row['month'] + 1
            target_year = row['year']
    return pd.Series({'target_month': target_month, 'target_year': target_year})

# Apply to get target contract month and year
pnw_flat_corn[['target_month', 'target_year']] = pnw_flat_corn.apply(get_spot_contract_info, axis=1)

# Filter for rows where contract_month and contract_year match target
spot_pnw_flat_corn = pnw_flat_corn[(pnw_flat_corn['contract_month'] == pnw_flat_corn['target_month']) & 
             (pnw_flat_corn['contract_year'] == pnw_flat_corn['target_year'])]

# Drop helper columns if not needed
spot_pnw_flat_corn = spot_pnw_flat_corn.drop(columns=['day', 'month', 'year', 'target_month', 'target_year'])

In [0]:
spot_pnw_corn_flat_df=spot_pnw_flat_corn.rename(columns={'value':'FOB_PNW'})
spot_usg_flat_corn_df=spot_usg_flat_corn.rename(columns={'value':'FOB_USG'})
flat_corn_upbb_df=flat_corn_upbb.rename(columns={'UPBB':'FOB_UPBB'})

spot_pnw_corn_flat_df=spot_pnw_corn_flat_df[['date','FOB_PNW','virtual_date']]
spot_usg_flat_corn_df=spot_usg_flat_corn_df[['date','FOB_USG']]
flat_corn_upbb_df=flat_corn_upbb_df[['date','FOB_UPBB','Season']]

fob_merged_flat=pd.merge(spot_pnw_corn_flat_df,spot_usg_flat_corn_df,on='date')
fob_merged_flat=pd.merge(fob_merged_flat,flat_corn_upbb_df,on='date')
fob_merged_flat


### CURVES

In [0]:
import plotly.graph_objects as go


freight_viet_flat=pd.merge(spot_us_viet,spot_arg_viet,on='date',how='inner')
freight_viet_flat=freight_viet_flat[['date','arg_to_viet','us_to_viet','virtual']]
freight_viet_flat['spread']=(freight_viet_flat['arg_to_viet']-freight_viet_flat['us_to_viet'])/0.3937

viet_w_freight_flat=pd.merge(flat_corn_upbb,freight_viet_flat,on='date',how='inner')
viet_w_freight_flat['arg_eq_us']=viet_w_freight_flat['UPBB']+viet_w_freight_flat['spread']


viet_w_freight_flat=viet_w_freight_flat[['date','UPBB','arg_eq_us','virtual','spread','Season']]
viet_w_freight_flat_w_us=pd.merge(viet_w_freight_flat,spot_pnw_flat_corn,on='date',how='inner')

viet_w_freight_flat_w_us['final_spread']=viet_w_freight_flat_w_us['arg_eq_us']-viet_w_freight_flat_w_us['value']

viet_flat = go.Figure()

seasons = viet_w_freight_flat_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season
    df_season = viet_w_freight_flat_w_us[viet_w_freight_flat_w_us['Season'] == season]

    # Final spread (left axis)
    viet_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    viet_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'US PNW ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    viet_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
viet_flat.update_layout(
    title='Arg Corn to viet_flatnam — Spread vs US PNW by Season',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (USD)', side='left'),
    yaxis2=dict(
        title='US PNW Value (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

viet_flat.show()
viet_flat_html = viet_flat.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
spot_us_kor

In [0]:
freight_kor_flat=pd.merge(spot_us_kor,spot_arg_kor,on='date',how='inner')
freight_kor_flat=freight_kor_flat[['date','arg_to_kor','us_to_kor','virtual']]
freight_kor_flat['spread']=(freight_kor_flat['arg_to_kor']-freight_kor_flat['us_to_kor'])/0.3937

kor_w_freight_flat=pd.merge(flat_corn_upbb,freight_kor_flat,on='date',how='inner')
kor_w_freight_flat['arg_eq_us']=kor_w_freight_flat['UPBB']+kor_w_freight_flat['spread']


kor_w_freight_flat=kor_w_freight_flat[['date','UPBB','arg_eq_us','virtual','spread','Season']]
kor_w_freight_flat_w_us=pd.merge(kor_w_freight_flat,spot_pnw_flat_corn,on='date',how='inner')

kor_w_freight_flat_w_us['final_spread']=kor_w_freight_flat_w_us['arg_eq_us']-kor_w_freight_flat_w_us['value']



In [0]:
freight_kor_flat=pd.merge(spot_us_kor,spot_arg_kor,on='date',how='inner')
freight_kor_flat=freight_kor_flat[['date','arg_to_kor','us_to_kor','virtual']]
freight_kor_flat['spread']=(freight_kor_flat['arg_to_kor']-freight_kor_flat['us_to_kor'])/0.3937

kor_w_freight_flat=pd.merge(flat_corn_upbb,freight_kor_flat,on='date',how='inner')
kor_w_freight_flat['arg_eq_us']=kor_w_freight_flat['UPBB']+kor_w_freight_flat['spread']


kor_w_freight_flat=kor_w_freight_flat[['date','UPBB','arg_eq_us','virtual','spread','Season']]
kor_w_freight_flat_w_us=pd.merge(kor_w_freight_flat,spot_pnw_flat_corn,on='date',how='inner')

kor_w_freight_flat_w_us['final_spread']=kor_w_freight_flat_w_us['arg_eq_us']-kor_w_freight_flat_w_us['value']

kor_flat = go.Figure()

seasons = kor_w_freight_flat_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season

    df_season = kor_w_freight_flat_w_us[kor_w_freight_flat_w_us['Season'] == season]

    # Final spread (left axis)
    kor_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    kor_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'US PNW ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    kor_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
kor_flat.update_layout(
    title='Arg Corn to Korea — Spread vs US PNW by Season',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (USD)', side='left'),
    yaxis2=dict(
        title='US PNW Value (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB (USD)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

kor_flat.show()
kor_flat_html = kor_flat.to_html(include_plotlyjs='cdn', full_html=True)

In [0]:
freight_egy_flat=pd.merge(spot_us_egy,spot_arg_egy,on='date',how='inner')
freight_egy_flat=freight_egy_flat[['date','arg_to_egy','us_to_egy','virtual']]
freight_egy_flat['spread']=(freight_egy_flat['arg_to_egy']-freight_egy_flat['us_to_egy'])/0.3937

egy_w_freight_flat=pd.merge(flat_corn_upbb,freight_egy_flat,on='date',how='inner')
egy_w_freight_flat['arg_eq_us']=egy_w_freight_flat['UPBB']+egy_w_freight_flat['spread']


egy_w_freight_flat=egy_w_freight_flat[['date','UPBB','arg_eq_us','virtual','spread','Season']]
egy_w_freight_flat_w_us=pd.merge(egy_w_freight_flat,spot_usg_flat_corn,on='date',how='inner')

egy_w_freight_flat_w_us['final_spread']=egy_w_freight_flat_w_us['arg_eq_us']-egy_w_freight_flat_w_us['value']

egy_flat = go.Figure()



egy_flat = go.Figure()

seasons = egy_w_freight_flat_w_us['Season'].unique()

# Loop through each season to plot both lines
for season in seasons:
    if season == '2022/2023':
        continue  # Skip this season
    df_season = egy_w_freight_flat_w_us[egy_w_freight_flat_w_us['Season'] == season]

    # Final spread (left axis)
    egy_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['final_spread'],
        mode='lines',
        name=f'Final Spread ({season})',
        yaxis='y1'
    ))

    # Value (right axis)
    egy_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['value'],
        mode='lines',
        name=f'USG ({season})',
        yaxis='y2',
        line=dict(dash='dot')  # Dashed lines to distinguish from spread
    ))

        # Value (right axis)
    egy_flat.add_trace(go.Scatter(
        x=df_season['virtual_date'],
        y=df_season['UPBB'],
        mode='lines',
        name=f'FOB UPBB ({season})',
        yaxis='y3',
        line=dict(dash='dash')  # Dashed lines to distinguish from spread
    ))

# Update layout with dual y-axis
egy_flat.update_layout(
    title='Arg Corn VS FLAT USG to Egypt (El Dekheila)',
    xaxis=dict(title='Date',tickformat="%b %d"),
    yaxis=dict(title='Final Spread (cts/bu)', side='left'),
    yaxis2=dict(
        title='USG Value',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    yaxis3=dict(
        title='FOB UPBB',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)'),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

egy_flat.show()
egy_flat_html = egy_flat.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:
egy_w_freight_flat_w_us=egy_w_freight_flat_w_us.rename(columns={'final_spread': 'corn_to_egy'})
viet_w_freight_flat_w_us=viet_w_freight_flat_w_us.rename(columns={'final_spread': 'corn_to_viet'})
kor_w_freight_flat_w_us=kor_w_freight_flat_w_us.rename(columns={'final_spread': 'corn_to_kor'})

viet_w_freight_flat_w_us_small=viet_w_freight_flat_w_us[['date','corn_to_viet']]
egy_w_freight_flat_w_us_small=egy_w_freight_flat_w_us[['date','corn_to_egy']]
kor_w_freight_flat_w_us_small=kor_w_freight_flat_w_us[['date','corn_to_kor','virtual_date','Season']]

dest_merged_flat=pd.merge(viet_w_freight_flat_w_us_small,egy_w_freight_flat_w_us_small,on='date',how='inner')
dest_merged_flat=pd.merge(dest_merged_flat,kor_w_freight_flat_w_us_small,on='date',how='inner')

fob_merged_filt=fob_merged[['date','FOB_UPBB','FOB_USG','FOB_PNW']]
dest_w_fob_flat=pd.merge(dest_merged_flat,fob_merged_filt,on='date',how='inner')


# Define line styles for each variable
line_styles = {
    'corn_to_viet': 'solid',
    'corn_to_egy': 'dot',
    'corn_to_kor': 'dash',
    'FOB_UPBB': 'solid',
    'FOB_USG': 'dot',
    'FOB_PNW': 'dash'
}

# Separate by y-axis
spread_vars = ['corn_to_viet', 'corn_to_egy', 'corn_to_kor']
fob_vars = ['FOB_UPBB', 'FOB_USG', 'FOB_PNW']

# Assign a color to each season (excluding 2022/2023)
season_list = dest_w_fob_flat['Season'].unique()
season_colors = {
    season: px.colors.qualitative.Plotly[i % len(px.colors.qualitative.Plotly)]
    for i, season in enumerate(season_list) if season != '2022/2023'
}

# Initialize figure
destinations = go.Figure()

fob_colors = {
    'FOB_UPBB': 'black',
    'FOB_USG': 'red',
    'FOB_PNW': 'yellow'
}

# Add traces
for season in season_list:
    if season == '2022/2023':
        continue
    dest_w_fob_flat_season = dest_w_fob_flat[dest_w_fob_flat['Season'] == season]
    color = season_colors[season]

    # Add spread variables to left y-axis using season colors
    for var in spread_vars:
        destinations.add_trace(go.Scatter(
            x=dest_w_fob_flat_season['virtual_date'],
            y=dest_w_fob_flat_season[var],
            name=f"{var.replace('corn_to_', '').capitalize()} ({season})",
            line=dict(color=color, dash=line_styles[var]),
            yaxis='y1'
        ))

    # Add FOB variables to right y-axis using fixed colors per FOB variable
    for var in fob_vars:
        destinations.add_trace(go.Scatter(
            x=dest_w_fob_flat_season['virtual_date'],
            y=dest_w_fob_flat_season[var],
            name=f"{var} ({season})",
            line=dict(color=fob_colors[var], dash=line_styles[var]),
            yaxis='y2'
        ))

# Layout
destinations.update_layout(
    title='Corn Spreads and FOB Values by Season',
    xaxis=dict(title='Virtual Date', tickformat="%b %d"),
    yaxis=dict(title='Spread (cts/bu)', side='left'),
    yaxis2=dict(
        title='FOB Price (USD/mt)',
        overlaying='y',
        side='right',
        showgrid=False
    ),
    template='plotly_white',
    legend=dict(x=1.01, y=1, bgcolor='rgba(255,255,255,0)', borderwidth=0),
    hovermode='x unified',
    margin=dict(l=60, r=60, t=50, b=40)
)

destinations.show()
destinations_html = destinations.to_html(include_plotlyjs='cdn', full_html=True)


In [0]:

flat_recap  = f"""
<!DOCTYPE html>
<html>
<head>
    <title>Corn to Destinations, flat prices</title>
    <style>
        body {{
            font-family: Arial, egyns-serif;
            padding: 20px;
        }}
        h2 {{
            margin-top: 40px;
            color: #2c3e50;
        }}
        .chart-container {{
            margin-bottom: 50px;
        }}
    </style>
</head>
<body>
    <h1>Corn to Egypt, ARG VS US</h1>
    <h2>Premiums</h2>
    <div class="chart-container">{egy_html}</div>
    <h2>Flat price</h2>
    <div class="chart-container">{egy_flat_html}</div>
    
    <h1>Corn to Vietnam, ARG VS US</h1>
    <h2>Premiums</h2>
    <div class="chart-container">{viet_html}</div>
    <h2>Flat price</h2>
    <div class="chart-container">{viet_flat_html}</div>

    <h1>Corn to Korea, ARG VS US</h1>
    <h2>Premiums</h2>
    <div class="chart-container">{kor_html}</div>
    <h2>Flat price</h2>
    <div class="chart-container">{kor_flat_html}</div>

</body>
</html>
"""


flat_report_bytes = flat_recap.encode("utf-8")


test=['florian.girardi-ext@ldc.com','gonzalo.lascombes@ldc.com','valentin.chiesa@ldc.com']
test_2=['florian.girardi-ext@ldc.com']
      
# Send email with embedded chart and table
LDCDataAccessLayerPy.mail.mail_send(
    to=test_2,
    subject=f'Corn to dest Flat Prices 2 {datetime.now().strftime("%d-%m")}',
    from_addr="florian.girardi-ext@ldc.com",
    mime_type="html",
    body="Please find the attached report.", attachment={"Recap_Values.html": flat_report_bytes}
)

